<a href="https://colab.research.google.com/github/Saakinaa/Sentiment_Analysis/blob/main/_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install kaggle

In [ ]:
# configuring the path of the Kaggle.json file
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# **Importing Twitter Sentiment Analysis Dataset**

In [ ]:
# API to fetch the dataset from Kaggle
! kaggle datasets download -d kazanova/sentiment140

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
sentiment140.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
! kaggle datasets download -d mdismielhossenabir/sentiment-analysis
# kaggle datasets download -d mdismielhossenabir/sentiment-analysis/data

Dataset URL: https://www.kaggle.com/datasets/mdismielhossenabir/sentiment-analysis
License(s): MIT
sentiment-analysis.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
# Extract the zip file
from zipfile import ZipFile
dataset = '/content/sentiment140.zip'

with ZipFile(dataset,'r') as zip:
  zip.extractall()
  print('The dataset is extracted')

The dataset is extracted


# **Importing Libraries**

In [ ]:
import numpy as np
import pandas as pd
import re #stands for regular expressions
import seaborn as sns
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# **Preprocessing**

In [ ]:
import nltk
nltk.download('stopwords') # Will download

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# Printing the stopwords in english
print(stopwords.words('english')) #In order to reduce the size and complexity of our data. doesn't really have such meaning

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [ ]:
data = pd.read_csv("training.1600000.processed.noemoticon.csv", encoding = "ISO-8859-1", header=None)

In [ ]:
# Display our dataset
data

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
...,...,...,...,...,...,...
1599995,4,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...
1599996,4,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...
1599997,4,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...
1599998,4,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...


In [ ]:
# Check the number of rows and columns
data.shape

(1600000, 6)

In [ ]:
# Print first 5 rows of our dataframe
data.head()

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [ ]:
# Naming the columns and reading dataset

#First way to do:
data.columns = ['target','ids','date','flag','user','text']

#Second way to do:
columns_names = ['target','ids','date','flag','user','text']
data = pd.read_csv("training.1600000.processed.noemoticon.csv", encoding = "ISO-8859-1", header=None, names=columns_names)


In [ ]:
# Display dataset
data.head()

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [ ]:
# Check for missing values in the dataset
data.isnull().sum() #we can observe that there is no missing values in our dataset

,0
target,0
ids,0
date,0
flag,0
user,0
text,0


In [ ]:
# Distribution of the target column
data['target'].value_counts() #Equal distribution

,count
target,
0,800000
4,800000


The reason to perfom this operation is that when it's not equally distributed, we have to do upsamlping and downsapling to make this distribution.
So if we have to work with this kind of distribution ... (Check this)

## **Handle Target**

In [ ]:
# Convert the target column 4 to 1

# First way
data['target'] = data['target'].replace(4,1)

# Second way
data['target'] = data['target'].apply(lambda x: 1 if x == 4 else x)

# Third way
data.replace({'target':{4:1}}, inplace = True)

# Verify
# Distribution of the target column
data['target'].value_counts() #Equal distribution

,count
target,
0,800000
1,800000


For now :
- 0 stands for Negative tweet
- 1 stands for Positive tweet

### **Stemming**

Process of reducing word to its Root word

**Example**:

actor, actress, acting = act

In [ ]:
port_stem = PorterStemmer()

In [ ]:
def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]', ' ', content) # substitute
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content)
  #return this stemmed_content
  return stemmed_content

#content here will correspond to the text column


In [ ]:
data['stemming_content'] = data['text'].apply(stemming) # 15 minutes to complete
#require huge time to complete because there are 1.6 million rows

In [ ]:
data.head() # we will notice that there is a new column called stemming_content in our dataframe


,target,ids,date,flag,user,text,stemming_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see


In [ ]:
# Display the stemming content column
print(data['stemming_content'])

0          switchfoot http twitpic com zl awww bummer sho...
1          upset updat facebook text might cri result sch...
2          kenichan dive mani time ball manag save rest g...
3                            whole bodi feel itchi like fire
4                              nationwideclass behav mad see
                                 ...                        
1599995                           woke school best feel ever
1599996    thewdb com cool hear old walt interview http b...
1599997                         readi mojo makeov ask detail
1599998    happi th birthday boo alll time tupac amaru sh...
1599999    happi charitytuesday thenspcc sparkschar speak...
Name: stemming_content, Length: 1600000, dtype: object


In [ ]:
# Display the stemming content column
print(data['target'])

0          0
1          0
2          0
3          0
4          0
          ..
1599995    1
1599996    1
1599997    1
1599998    1
1599999    1
Name: target, Length: 1600000, dtype: int64


# **Build Model**

In [ ]:
# We don't really need id, date, flag and the user's name columns to know either it's a positive or negative comment
# We will just save stem and target columns
# Define:
X = data['stemming_content'].values
y = data['target']

In [ ]:
print(X)

['switchfoot http twitpic com zl awww bummer shoulda got david carr third day'
 'upset updat facebook text might cri result school today also blah'
 'kenichan dive mani time ball manag save rest go bound' ...
 'readi mojo makeov ask detail'
 'happi th birthday boo alll time tupac amaru shakur'
 'happi charitytuesday thenspcc sparkschar speakinguph h']


In [ ]:
print(y)

0          0
1          0
2          0
3          0
4          0
          ..
1599995    1
1599996    1
1599997    1
1599998    1
1599999    1
Name: target, Length: 1600000, dtype: int64


In [ ]:
# Splitting data into training data and testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=2)

In [ ]:
# Training and Testing data shape
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1280000,), (320000,), (1280000,), (320000,))

In [ ]:
# Print out X_train
X_train

array(['watch saw iv drink lil wine', 'hatermagazin',
       'even though favourit drink think vodka coke wipe mind time think im gonna find new drink',
       ..., 'eager monday afternoon',
       'hope everyon mother great day wait hear guy store tomorrow',
       'love wake folger bad voic deeper'], dtype=object)

In [ ]:
# Print out X_test
X_test
original = X_test
print(X_test)

  (0, 15110)	0.1719352837797837
  (0, 31168)	0.1624772418052177
  (0, 67828)	0.26800375270827315
  (0, 106069)	0.36555450010904555
  (0, 132364)	0.255254889555786
  (0, 138164)	0.23688292264071406
  (0, 171378)	0.2805816206356074
  (0, 271016)	0.45356623916588285
  (0, 279082)	0.17825180109103442
  (0, 388348)	0.2198507607206174
  (0, 398906)	0.34910438732642673
  (0, 409143)	0.3143047059807971
  (0, 420984)	0.17915624523539805
  (1, 6463)	0.30733520460524466
  (1, 15110)	0.211037449588008
  (1, 145393)	0.575262969264869
  (1, 217562)	0.40288153995289894
  (1, 256777)	0.28751585696559306
  (1, 348135)	0.4739279595416274
  (1, 366203)	0.24595562404108307
  (2, 22532)	0.3532582957477176
  (2, 34401)	0.37916255084357414
  (2, 89448)	0.36340369428387626
  (2, 183312)	0.5892069252021465
  (2, 256834)	0.2564939661498776
  :	:
  (319994, 443794)	0.2782185641032538
  (319995, 107868)	0.33399349737546963
  (319995, 109379)	0.3020896484890833
  (319995, 155493)	0.2770682832971669
  (319995, 2133

In [ ]:
# ML model doesn't understand text data, so we will convert these values to numerical type
vectorizer = TfidfVectorizer()
vectorizer.fit(X_train)

X_train = vectorizer.transform(X_train)
X_test = vectorizer.transform(X_test)

In [ ]:
# Print out X_train
X_train

<1280000x461488 sparse matrix of type '<class 'numpy.float64'>'
	with 9453092 stored elements in Compressed Sparse Row format>

In [ ]:
# Print out X_test
X_test

<320000x461488 sparse matrix of type '<class 'numpy.float64'>'
	with 2289192 stored elements in Compressed Sparse Row format>

## **Training Machine Learning Model**

In [ ]:
# Defining our prediction model
model = LogisticRegression(max_iter=1000)

In [ ]:
# Fit our model to our data
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
# Make prediction
predicted = model.predict(X_test)

## **Model Evaluation**

In [ ]:
# Accuracy score on the training data
training_data_accuracy = accuracy_score(model.predict(X_train), y_train)
# Print accuracy on the training data
print('Accuracy score on the training data : ', training_data_accuracy) # accuracy de 81%. sur 100 tweets, 81 seront bien classes

Accuracy score on the training data :  0.79871953125


In [ ]:
# Accuracy score on the test data
test_data_accuracy = accuracy_score(predicted, y_test)
# Print accuracy on the test data
print('Accuracy score on the test data : ', test_data_accuracy)
# Our model accuracy de 78%. sur 100 tweets, 78 seront normalement classes

Accuracy score on the test data :  0.77668125


## **Saving this model**

In [ ]:
import pickle

In [ ]:
filename = 'trained_model.sav'
pickle.dump(model, open(filename, 'wb')) # Check the wb

In [ ]:
# Loading the saving model
loaded_model = pickle.load(open('trained_model.sav', 'rb'))

## **Test Model**

In [ ]:
# Get the index corresponding to the 200th row of X_test
index_100 = y_test.index[100]

# Use the index to access elements in X_test and y_test
X_new = X_test[100]
print(y_test.loc[index_100]) # Use .loc to access by label/index

prediction = model.predict(X_new)
print(prediction)

0
[0]


In [ ]:
# Get the index corresponding to the 200th row of X_test
index_200 = y_test.index[200]

# Use the index to access elements in X_test and y_test
X_new = X_test[200]
print(y_test.loc[index_200]) # Use .loc to access by label/index

prediction = model.predict(X_new)
print(prediction)

1
[1]


In [ ]:
# If Statement
if prediction[0] == 0:
  print('The tweet is Negative')
else:
  print('The tweet is Positive')

The tweet is Positive


In [ ]:
# Get the original text we had before encoding to check either it's positive or negative
#original_text = X_test[200]
original_Text = original[200]
print("Original Text:", original_Text)

Original Text:   (0, 93795)	0.18766071909970514
  (0, 127090)	0.32853785623524473
  (0, 145988)	0.18586475941913905
  (0, 242268)	0.20103456365786704
  (0, 326966)	0.3047343739770557
  (0, 372988)	0.3474823166340872
  (0, 387466)	0.46923185113800253
  (0, 400002)	0.33425960056335163
  (0, 425173)	0.4861797169777241


In [ ]:
# Print out 200
# X_new = X_test[200]
# print(y_test[200])
# prediction = model.predict(X_new)
# print(prediction)

In [ ]:
# Assuming you kept the raw text data
#original_text = X_test[3]  # Retrieve the original text
#print("Original Text:", original_text)

Original Text:   (0, 120049)	0.5801692086479822
  (0, 142563)	0.5742504216907061
  (0, 241674)	0.57761591263124


# **Part 2: Sentiment Analysis**

In [ ]:
import pandas as pd
import numpy as np
data = pd.read_csv('sentiment_analysis.csv')
# data
# data.isnull().sum()

In [ ]:
data

,Year,Month,Day,Time of Tweet,text,sentiment,Platform
0,2018,8,18,morning,What a great day!!! Looks like dream.,positive,Twitter
1,2018,8,18,noon,"I feel sorry, I miss you here in the sea beach",positive,Facebook
2,2017,8,18,night,Don't angry me,negative,Facebook
3,2022,6,8,morning,We attend in the class just for listening teac...,negative,Facebook
4,2022,6,8,noon,"Those who want to go, let them go",negative,Instagram
...,...,...,...,...,...,...,...
494,2015,10,18,night,"According to , a quarter of families under six...",negative,Twitter
495,2021,2,25,morning,the plan to not spend money is not going well,negative,Instagram
496,2022,5,30,noon,uploading all my bamboozle pictures of facebook,neutral,Facebook
497,2018,8,10,night,congratulations ! you guys finish a month ear...,positive,Twitter


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499 entries, 0 to 498
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Year           499 non-null    int64 
 1   Month          499 non-null    int64 
 2   Day            499 non-null    int64 
 3   Time of Tweet  499 non-null    object
 4   text           499 non-null    object
 5   sentiment      499 non-null    object
 6   Platform       499 non-null    object
dtypes: int64(3), object(4)
memory usage: 27.4+ KB


In [ ]:
data['sentiment'].unique()
data['Time of Tweet'].unique()
data['Platform'].unique()

array([' Twitter  ', ' Facebook ', 'Facebook', ' Instagram ', ' Twitter '],
      dtype=object)

In [ ]:
# Handle Platforms name
data['Platform'] = data['Platform'].replace({' Twitter ': 'Twitter', ' Twitter. ': 'Twitter', ' Facebook ': 'Facebook',  ' Instagram ': 'Instagram'})
data['Platform'].unique()

array(['Twitter', 'Facebook', 'Instagram'], dtype=object)

In [ ]:
# Encode the sentiment column
data['sentiment'] = data['sentiment'].replace({'positive': 1, 'negative': -1, 'neutral' : 0})
# Encode the time column
data['Time of Tweet'] = data['Time of Tweet'].replace({'morning': 6, 'noon': 12, 'night' : 18})
# Encode Platform column
data['Platform'] = data['Platform'].replace({'Twitter': 1, 'Facebook': 2, 'Instagram': 3})


<ipython-input-25-c8f234d0c254>:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Platform'] = data['Platform'].replace({'Twitter': 1, 'Facebook': 2, 'Instagram': 3})


In [ ]:
data

,Year,Month,Day,Time of Tweet,text,sentiment,Platform
0,2018,8,18,6,What a great day!!! Looks like dream.,1,1
1,2018,8,18,12,"I feel sorry, I miss you here in the sea beach",1,2
2,2017,8,18,18,Don't angry me,-1,2
3,2022,6,8,6,We attend in the class just for listening teac...,-1,2
4,2022,6,8,12,"Those who want to go, let them go",-1,3
...,...,...,...,...,...,...,...
494,2015,10,18,18,"According to , a quarter of families under six...",-1,1
495,2021,2,25,6,the plan to not spend money is not going well,-1,3
496,2022,5,30,12,uploading all my bamboozle pictures of facebook,0,2
497,2018,8,10,18,congratulations ! you guys finish a month ear...,1,1


In [ ]:
import numpy as np
import pandas as pd
import re #satnds for regular expressions
import seaborn as sns
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import nltk
nltk.download('stopwords') # it will download

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
# Printing the stopwords in english
print(stopwords.words('english')) #in order to reduce the size and complexity of our data. doesn't really have such meaning

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [ ]:
port_stem = PorterStemmer()

In [ ]:
def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]', ' ', content) # substitute
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content)
  #return this stemmed_content
  return stemmed_content

In [ ]:
data['stemming_content'] = data['text'].apply(stemming) # 15 minutes to complete
# Diplay Stemming Content
print(data['stemming_content'])

0                           great day look like dream
1                           feel sorri miss sea beach
2                                               angri
3      attend class listen teacher read slide nonsenc
4                                      want go let go
                            ...                      
494            accord quarter famili six live poverti
495                          plan spend money go well
496                   upload bamboozl pictur facebook
497             congratul guy finish month earli booo
498                        actual wish back taho miss
Name: stemming_content, Length: 499, dtype: object


In [ ]:
# Display the stemming content column
print(data['sentiment'])

0      1
1      1
2     -1
3     -1
4     -1
      ..
494   -1
495   -1
496    0
497    1
498   -1
Name: sentiment, Length: 499, dtype: int64


In [ ]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import nltk

# Ensure stopwords are downloaded
nltk.download('stopwords')

# Load dataset
data = pd.read_csv("sentiment_analysis.csv", encoding="ISO-8859-1")

# Clean Platform column
data['Platform'] = data['Platform'].replace({
    ' Twitter ': 'Twitter',
    ' Twitter. ': 'Twitter',
    ' Facebook ': 'Facebook',
    ' Instagram ': 'Instagram'
})

# Encode the sentiment and keep original sentiments for reference
data['sentiment_original'] = data['sentiment']
data['sentiment'] = data['sentiment'].replace({'positive': 1, 'negative': -1, 'neutral': 0})

# Text Preprocessing
port_stem = PorterStemmer()

def stemming(content):
    content = re.sub('[^a-zA-Z]', ' ', content)  # Remove non-alphabetic characters
    content = content.lower().split()
    content = [port_stem.stem(word) for word in content if word not in stopwords.words('english')]
    return ' '.join(content)

data['stemming_content'] = data['text'].apply(stemming)

# Define features (X) and target (y)
X = data['stemming_content'].values
y = data['sentiment']

# Split data into train and test sets, including original text
X_train, X_test, y_train, y_test, original_text_train, original_text_test = train_test_split(
    X, y, data['text'], test_size=0.2, stratify=y, random_state=42
)

# Now you have the original text for both train and test sets
original_text = original_text_test # Use the test set for later analysis

# Transform text data using TfidfVectorizer
vectorizer = TfidfVectorizer()
vectorizer.fit(X_train)
X_train = vectorizer.transform(X_train)
X_test = vectorizer.transform(X_test)

# Train the SVC model
svc = SVC(kernel='linear')
svc.fit(X_train, y_train)

# Make predictions
y_pred = svc.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of the model: {accuracy:.2f}")

# Confusion matrix and classification report
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Manual verification of predicted vs. actual sentiment with original text
results = pd.DataFrame({
    'Original Text': original_text,
    'Actual Sentiment': y_test.map({1: 'positive', -1: 'negative', 0: 'neutral'}).values,
    'Predicted Sentiment': pd.Series(y_pred).map({1: 'positive', -1: 'negative', 0: 'neutral'}).values
})

# Display a few results for verification
print("Sample of Predictions vs Actual Sentiments:")
print(results.head(10))

# Optional: Save results to a CSV for manual review
results.to_csv("sentiment_predictions.csv", index=False)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
<ipython-input-38-f4cc6d32225f>:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['sentiment'] = data['sentiment'].replace({'positive': 1, 'negative': -1, 'neutral': 0})


Accuracy of the model: 0.69
Confusion Matrix:
[[11 13  3]
 [ 1 35  4]
 [ 1  9 23]]

Classification Report:
              precision    recall  f1-score   support

          -1       0.85      0.41      0.55        27
           0       0.61      0.88      0.72        40
           1       0.77      0.70      0.73        33

    accuracy                           0.69       100
   macro avg       0.74      0.66      0.67       100
weighted avg       0.73      0.69      0.68       100


Sample of Predictions vs Actual Sentiments:
                                         Original Text Actual Sentiment  \
73                                          Soooo high          neutral   
322  Before I get too distracted, I`d like to thank...         positive   
425   Si, no bueno  I guess I just don`t entertain ...         negative   
377  Omg Wango Tango was **** AWSOME! I love my bab...         positive   
74                                         Both of you          neutral   
230  The exceptio